# Credit Line Review Assistant — SCS 3253 Term Project

Report outline this notebook mirrors: Objective -> Data Preparation -> Model Design -> Model Evaluation -> Conclusions.
See `docs/credit-line-review-assistant-proposal.md` for the full design.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import plotly.express as px

from credit_line_review.data import load_cards, load_transactions, load_users

## 1. Objective

We predict a data-driven recommended credit limit for **existing** cardholders from their demographics and observed spending behavior, then flag accounts whose actual limit diverges sharply from that recommendation. This is a retrospective credit-line-review tool, not new-account underwriting.

## 2. Data Preparation

### 2.1 Load raw data

In [2]:
users = load_users()
cards = load_cards()
transactions = load_transactions(nrows=5000)

print("users:", users.shape)
print("cards:", cards.shape)
print("transactions (sample):", transactions.shape)

users: (2000, 14)
cards: (6146, 13)
transactions (sample): (5000, 12)


### 2.2 Does `card_type` affect whether `credit_limit` is meaningful?

A debit card shouldn't carry a real credit limit as a business concept. Check the split before deciding whether to filter.

In [3]:
print(cards["card_type"].value_counts())
print(cards.groupby("card_type")["credit_limit"].describe())

card_type
Debit              3511
Credit             2057
Debit (Prepaid)     578
Name: count, dtype: int64
                  count          mean           std  min      25%      50%  \
card_type                                                                    
Credit           2057.0  11174.380165   6834.713404  0.0   7200.0  10100.0   
Debit            3511.0  18557.888636  12966.142501  0.0  11010.0  16454.0   
Debit (Prepaid)   578.0     64.448097     24.687894  0.0     51.0     65.0   

                     75%       max  
card_type                           
Credit           13900.0   98100.0  
Debit            23478.0  151223.0  
Debit (Prepaid)     80.0     145.0  


If `credit_limit` is populated for Debit cards too, that confirms a data-quality quirk: restrict modeling to `card_type == "Credit"` so the regression target only reflects an actual credit product.

In [4]:
credit_cards = cards[cards["card_type"] == "Credit"]
print("credit cards:", credit_cards.shape, "of", cards.shape[0], "total cards")

credit cards: (2057, 13) of 6146 total cards


### 2.3 Distribution of the regression target and key predictors

Financial quantities (income, debt, credit limit) are usually right-skewed / log-normal. Check skew before deciding whether to log-transform for the linear model.

In [5]:
monetary_cols = {
    "credit_limit": credit_cards["credit_limit"],
    "yearly_income": users["yearly_income"],
    "total_debt": users["total_debt"],
}
for name, series in monetary_cols.items():
    print(f"{name}: skew={series.skew():.2f}, mean={series.mean():.0f}, median={series.median():.0f}")
    fig = px.histogram(series, nbins=50, title=f"Distribution of {name}")
    fig.show()

credit_limit: skew=3.20, mean=11174, median=10100


yearly_income: skew=3.45, mean=45716, median=40744


total_debt: skew=1.81, mean=63710, median=58251


In [6]:
log_limit = np.log1p(credit_cards["credit_limit"])
print(f"log1p(credit_limit): skew={log_limit.skew():.2f}")
fig = px.histogram(log_limit, nbins=50, title="Distribution of log1p(credit_limit)")
fig.show()

log1p(credit_limit): skew=-5.52


A skew close to 0 after `log1p` (vs. the raw skew printed above) would justify log-transforming `credit_limit`, `yearly_income`, and `total_debt` before fitting the linear model — but the log1p skew printed above is large and *negative*, the opposite of what a successful log-transform of a right-skewed variable should look like. Investigate why.

### 2.3.1 Why did `log1p(credit_limit)` get *more* skewed, not less?

A cluster of `credit_limit == 0` rows would sit far below the rest of the (log-transformed) distribution, dragging the skew negative. Check for that directly.

In [7]:
n_zero = (credit_cards["credit_limit"] == 0).sum()
print(f"Credit-card rows with credit_limit == 0: {n_zero} "
      f"({n_zero / len(credit_cards):.1%} of {len(credit_cards)} credit cards)")

Credit-card rows with credit_limit == 0: 26 (1.3% of 2057 credit cards)


These are almost certainly closed/frozen accounts, not a real "what should their limit be" case — a genuine credit-line-review candidate has a nonzero limit today. Rather than silently dropping or blindly modeling them, `build_modeling_table` (Task 4) tags them with an `is_zero_limit` flag: they stay in the exported table for reporting, but are excluded from the train/test split used to fit the regressions, and get their own "closed accounts" segment in the dashboard instead of an under/over-limited flag.

In [8]:
print("credit_score:", users["credit_score"].describe())
fig = px.histogram(users["credit_score"], nbins=40, title="Distribution of credit_score")
fig.show()

fig = px.histogram(transactions["amount"], nbins=50, title="Distribution of transaction amount (sample)")
fig.show()

credit_score: count    2000.000000
mean      709.734500
std        67.221949
min       480.000000
25%       681.000000
50%       711.500000
75%       753.000000
max       850.000000
Name: credit_score, dtype: float64


### 2.4 Feature engineering & modeling table

Aggregate transactions to one row per card, join to cards + users, log-transform monetary columns, and split by `client_id` so a client's cards never span train and test — implemented in `credit_line_review.features` (Tasks 3-4 of the implementation plan) and wired into this notebook in Task 10.

## 3. Model Design

### 3.1 Baseline (predict the training mean)
Not a course model - the floor any real model has to beat.

### 3.2 Linear/Ridge Regression
*Course: Module 5 - Training Models & Feature Selection.* Chosen for coefficient-level interpretability, which matters for a credit decision.

### 3.3 Random Forest Regression
*Course: Module 7 - Decision Trees & Ensemble Learning.* Chosen over SVR (Module 6) - handles non-linear feature interactions without kernel tuning and scales better to this row count.

### 3.4 KMeans persona clustering + PCA visualization
*Course: Module 4 - Clustering (KMeans) and Module 8 - Dimensionality Reduction (PCA).*

### 3.5 Beyond the course
Residual diagnostics, log-transforming skewed monetary columns, grouped train/test splitting, RFM-style behavioral features, and SHAP feature importance are not course topics - see `docs/credit-line-review-assistant-proposal.md` §5.2 for why each one is needed here.

## 4. Model Evaluation

*Course: Module 3 - Classification.* The regression-vs-actual residual, thresholded, becomes a binary "flag/don't flag" decision - precision/recall-style thinking from Module 3 applies to that decision even though the underlying models are regression/clustering, not classification.

## 5. Conclusions